# 03 — PDP + ConvNeXt-V2-Base: nghiệm thu adapter

**Chưa chạy.** Tạo ngày 09/10/2026. Phụ thuộc: notebook 01 foundation và notebook 02 full ResNet/PDP CUDA smoke đã PASS; môi trường Conda Python 3.10, CUDA, runtime pins và release real-only đã khóa. Restart kernel trước khi chạy để nạp code adapter mới. Không cài package trong notebook.

Mục tiêu: ConvNeXt-V2-Base pretrained riêng → detector core COCO chuyển có chọn lọc → feature maps/masks/projections ở 640/800 → losses/gradient/optimizer/freeze → checkpoint/config reload và Lightning resume. Giữ encoder/decoder 6/6, d_model 256, 300 queries, 4 feature levels, prompt 100×10, 225 outputs; LoRA/K=3/FSA/freeze bổ sung tắt.

[Backbone chính thức](https://huggingface.co/timm/convnextv2_base.fcmae_ft_in22k_in1k), [timm v0.9.12](https://github.com/huggingface/pytorch-image-models/blob/v0.9.12/timm/models/convnext.py). Timm trả 4 stage strides 4/8/16/32; adapter lấy 3 stage cuối theo metadata và detector tạo level thứ tư. Normalization kiểm tra cùng processor; không crop khay theo classifier transforms.

Weights public được tải vào cache **khi bạn chạy notebook**, có revision/SHA256; `offline=True` chỉ reuse cache. Baseline summary/run artifacts ở máy khác cần copy hoặc chỉ định path; không suy ra PASS bằng doc hoặc file notebook cũ. Raw/split/holdout giữ nguyên, chỉ dùng 2 ảnh train Task 1 và batch 1 FP32.

PASS của notebook là `PASS_CONVNEXT_ADAPTER_CUDA_SMOKE`. Không chứng nhận convergence, full 5 task, native kernel/Vast.ai, calibration hay đường CLI/runner. [Readiness training](../../docs/project/CONVNEXT_READINESS.md) ghi phần còn thiếu.

## 1. Setup và run metadata

Sửa roots hoặc baseline_summary nếu artifacts không cùng máy. Giữ model/revision mặc định để đối sánh; không thay split. `require_custom_kernel=False` cho phép PyTorch CUDA fallback và vẫn ghi native gate riêng. Chạy mỗi resolution tuần tự để giới hạn VRAM; nếu OOM kết quả là PARTIAL.

In [ ]:
import os, sys, json, uuid, subprocess, hashlib, shutil, gc, time
from pathlib import Path
from types import SimpleNamespace
from unittest.mock import patch
from importlib import metadata
REPO=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'pdp/main.py').exists())
for path in [REPO,REPO/'pdp',REPO/'tests']:
    if str(path) not in sys.path:sys.path.insert(0,str(path))
from autocheckout.io import load_json,save_json,md5_file
from autocheckout.model_acceptance import RunEvidence,now,validate_loading_info
from tools.download_pdp_pretrained import download_pretrained,DEFAULT_REVISION,sha256_file

RAW=Path(os.getenv('AUTOCHECKOUT_DATA_ROOT',str(REPO/'data/archive'))).resolve()
RELEASE=Path(os.getenv('AUTOCHECKOUT_RELEASE_ROOT',str(REPO/'data/processed/rpc_100-4x25_seed0_native_v1'))).resolve()
CONFIG={'seed':0,'resolutions':[640,800],'batch_size':1,'sample_count':2,'workers':0,
 'device':'cuda','precision':'32-true','tf32':False,'offline':False,'require_custom_kernel':False,
 'backbone':'convnextv2_base.fcmae_ft_in22k_in1k',
 'backbone_repo':'timm/convnextv2_base.fcmae_ft_in22k_in1k',
 'backbone_revision':'9e250ed8f88b436472d2b24af26f82a8aa8c719d',
 'detector_revision':DEFAULT_REVISION,
 'baseline_summary':os.getenv('AUTOCHECKOUT_BASELINE_SUMMARY',''),
 'raw_root':str(RAW),'release_root':str(RELEASE)}
assert CONFIG['batch_size']==1 and CONFIG['sample_count']==2
os.environ['USE_TF']='0'
os.environ['HF_HUB_OFFLINE']='1' if CONFIG['offline'] else '0'
os.environ.setdefault('OMP_NUM_THREADS','1');os.environ.setdefault('MKL_NUM_THREADS','1')
os.environ.setdefault('MPLBACKEND','Agg')
ARTIFACT=REPO/'runs/modeling/convnext_adapter'/str(uuid.uuid4())
REQUIRED=['environment','baseline_evidence','release_lock','weights','adapter_regression','cuda_640','cuda_800','sources_preserved']
RUN=RunEvidence(ARTIFACT,CONFIG,REQUIRED,pass_status='PASS_CONVNEXT_ADAPTER_CUDA_SMOKE')
RUN.data.update(purpose='ConvNeXt-V2-Base feature adapter and PDP CUDA smoke',
 git_commit=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip(),
 git_dirty=bool(subprocess.check_output(['git','status','--porcelain'],cwd=REPO,text=True).strip()),
 python=sys.version,executable=sys.executable,convnext_accepted=False,full_training_ready=False)
RUN.persist()
shutil.copy2(REPO/'notebooks/modeling/03_convnext_v2_base_acceptance.ipynb',ARTIFACT/'notebook_source.ipynb')
SOURCE_BEFORE={}
for relative in ['pdp/models/backbones.py','pdp/models/modeling_deformable_detr.py',
                 'pdp/full_baseline_acceptance.py','pdp/engine.py','requirements.txt']:
    SOURCE_BEFORE[str(REPO/relative)]=('sha256',sha256_file(REPO/relative))
    snapshot=ARTIFACT/'source'/relative;snapshot.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(REPO/relative,snapshot)
print('Run',ARTIFACT.name,'started',RUN.data['started_at'],'outputs',ARTIFACT)

## 2. Environment và bằng chứng baseline

Gate này cần summary thật của notebook 02, đủ các gate environment/foundation/release/pretrained/CUDA640/CUDA800/source PASS. Default đọc marker latest; kiểm tra status của summary và marker, không tin marker đơn lẻ. Nếu clone chỉ có docs/notebooks mà thiếu `runs/`, copy summary+gates và weights/release từ máy nghiệm thu hoặc chạy lại notebook 02 trong environment phù hợp trước.

In [ ]:
with RUN.gate('environment') as evidence:
    assert sys.version_info[:2]==(3,10),'Select Conda pdp Python 3.10'
    pins={'torch':'2.2.2','torchvision':'0.17.2','timm':'0.9.12','transformers':'4.37.2',
          'pytorch-lightning':'2.1.3','packaging':'24.2','typing-extensions':'4.13.2'}
    versions={name:metadata.version(name) for name in pins}
    assert all(versions[n].split('+')[0]==v for n,v in pins.items()),versions
    checked=subprocess.run([sys.executable,'-m','pip','check'],capture_output=True,text=True)
    (ARTIFACT/'pip_check.log').write_text(checked.stdout+checked.stderr)
    assert checked.returncode==0,checked.stdout+checked.stderr
    import numpy as np
    import torch,timm,pytorch_lightning as pl
    import engine,main as pdp_main
    import models.modeling_deformable_detr as detr
    from models.configuration_deformable_detr import DeformableDetrConfig
    from models.image_processing_deformable_detr import DeformableDetrImageProcessor
    from models.backbones import plan_detector_transfer
    from datasets.coco_hug import CocoDetection
    from autocheckout.taskcfg import TaskConfig
    from full_baseline_acceptance import exercise_baseline,state_digest
    from inference import predict_batch
    from torch.utils.data import Subset
    assert torch.cuda.is_available(),'CUDA GPU required for this notebook; no CPU acceptance substitution'
    assert 'build_feature_backbone' in detr.DeformableDetrConvEncoder.__init__.__code__.co_names,'Restart kernel to load new adapter'
    torch.set_num_threads(1)
    torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False
    native=detr.MultiScaleDeformableAttention is not None
    native_tested=False
    if native:
        from test_pdp_f11_kernel import test_kernel_matches_pytorch_forward_and_backward
        test_kernel_matches_pytorch_forward_and_backward();native_tested=True
    assert native_tested or not CONFIG['require_custom_kernel'],str(detr.KERNEL_LOAD_ERROR)
    evidence.update(packages=versions,pip_check_returncode=0,gpu=torch.cuda.get_device_name(0),
                    cuda_runtime=torch.version.cuda,native_kernel_accepted=native_tested,
                    kernel_fallback=not native,kernel_error=str(detr.KERNEL_LOAD_ERROR))

with RUN.gate('baseline_evidence') as evidence:
    if CONFIG['baseline_summary']:
        baseline_path=Path(CONFIG['baseline_summary']).expanduser().resolve()
    else:
        base=REPO/'runs/modeling/full_pdp_baseline'
        marker=load_json(base/'latest.json')
        baseline_path=base/marker['run_id']/'summary.json'
    baseline=load_json(baseline_path)
    expected=['environment','foundation','release_lock','pretrained_files','cuda_640','cuda_800','sources_preserved']
    assert baseline['status']=='PASS_FULL_PDP_CUDA_SMOKE'
    assert all(baseline['gates'].get(n,{}).get('status')=='PASS' for n in expected)
    assert baseline.get('full_model_640_800_forward_backward_accepted') is True
    assert baseline['gates']['release_lock']['protocol']==[100,25,25,25,25]
    evidence.update(run_id=baseline['run_id'],summary_sha256=sha256_file(baseline_path),
                    source_status=baseline['status'],native_kernel_accepted=baseline.get('custom_kernel_accepted',False))
    shutil.copy2(baseline_path,ARTIFACT/'baseline_summary.json')

## 3. Release lock và chọn batch

Task mapping và mọi split/task JSON phải khớp config lock. Holdout chỉ hash, không đọc làm batch. Dùng đúng native image root, không dùng `checkout_800` legacy. Release/root dữ liệu giữ độc lập với model weights.

In [ ]:
with RUN.gate('release_lock') as evidence:
    lock=load_json(REPO/'configs/data/rpc_100-4x25_seed0_native_v1.json')
    TASKS=RELEASE/'tasks/real'
    task_config=RELEASE/'task_config.json'
    TASK_CONFIG=TaskConfig.load(task_config)
    assert [t.num_classes for t in TASK_CONFIG.data_tasks]==[100,25,25,25,25]
    assert (TASK_CONFIG.num_slots,TASK_CONFIG.num_classes)==(224,225) and TASK_CONFIG.seed==0
    assert lock['coordinates']=='native_pixels' and not lock['synthetic_included']
    assert sha256_file(task_config)==lock['input']['task_config_sha256']
    assert sha256_file(TASKS/'manifest.json')==lock['task_manifest_sha256']
    for name,digest in lock['input']['source_sha256'].items():
        path=RAW/name;assert sha256_file(path)==digest;SOURCE_BEFORE[str(path)]=('sha256',digest)
    for name,digest in lock['split_md5'].items():
        path=RELEASE/'splits'/name;assert md5_file(path)==digest;SOURCE_BEFORE[str(path)]=('md5',digest)
    for name,row in load_json(TASKS/'manifest.json')['files'].items():
        path=TASKS/name;assert md5_file(path)==row['md5'];SOURCE_BEFORE[str(path)]=('md5',row['md5'])
    SOURCE_BEFORE[str(task_config)]=('sha256',sha256_file(task_config))
    SOURCE_BEFORE[str(TASKS/'manifest.json')]=('sha256',sha256_file(TASKS/'manifest.json'))
    canonical=load_json(TASKS/'train_task_1.json')
    assert all(0<=a['category_id']<100 for a in canonical['annotations'])
    images=sorted(canonical['images'],key=lambda r:r['id'])[:CONFIG['sample_count']]
    CONFIG['image_ids']=[r['id'] for r in images]
    for row in images:
        path=(RAW/row['file_name']).resolve();assert path.is_relative_to(RAW) and path.is_file()
        SOURCE_BEFORE[str(path)]=('sha256',sha256_file(path))
    evidence.update(protocol=[100,25,25,25,25],image_ids=CONFIG['image_ids'],checked_files=len(SOURCE_BEFORE))
    del canonical

## 4a. Regression sau thay đổi factory chung

Factory cũng phục vụ ResNet baseline. Chạy các tests contract/status, ResNet inference/parameter counts và CPU+CUDA Lightning fixture sau sửa để chặn regression trước ConvNeXt smoke. Đây là gate runtime mới, không dùng kết quả tests baseline cũ làm PASS. Cần 0 failures/errors/skips trong JUnit.


In [ ]:
with RUN.gate('adapter_regression') as evidence:
    import xml.etree.ElementTree as ET
    junit=ARTIFACT/'adapter_regression.xml'
    command=[sys.executable,'-m','pytest','tests/test_backbone_adapter.py','tests/test_model_acceptance.py',
        'tests/test_pdp_f9_inference.py','tests/test_pdp_f10_parameters.py','tests/test_full_baseline_acceptance.py',
        '-q','-o','addopts=','--tb=short','--junitxml',str(junit),
        '--basetemp',str(ARTIFACT/'regression_tmp')]
    environment=os.environ.copy()
    environment.update(USE_TF='0',HF_HUB_OFFLINE='1',TRANSFORMERS_OFFLINE='1',OMP_NUM_THREADS='1',MKL_NUM_THREADS='1')
    with (ARTIFACT/'adapter_regression.log').open('w') as log:
        result=subprocess.run(command,cwd=REPO,env=environment,stdout=log,stderr=subprocess.STDOUT,timeout=1800)
    assert result.returncode==0,'See adapter_regression.log'
    suites=list(ET.parse(junit).getroot().iter('testsuite'))
    counts={key:sum(int(s.get(key,0)) for s in suites) for key in ['tests','errors','failures','skipped']}
    assert counts['tests']>0 and counts['errors']==counts['failures']==counts['skipped']==0,counts
    evidence.update(command=command,exit_code=result.returncode,counts=counts,junit=str(junit))
    print(counts)


## 4. Hai nguồn pretrained tách riêng

Detector COCO SenseTime giữ revision đã dùng ở baseline. ConvNeXt-V2-Base dùng revision `9e250ed8f88b436472d2b24af26f82a8aa8c719d`, chỉ tải config và safetensors từ repo public của timm. Ghi hashes trước model construction; local file được timm nạp qua pretrained_cfg_overlay, không tìm lại weights qua Hub trong model.

Warm start: giữ encoder/decoder/query/bbox core tương thích; **khởi tạo mới input projections và classifier 225 outputs**, giữ pretrained ConvNeXt backbone, tạo prompt mới. Không nạp nguyên checkpoint ResNet lên ConvNeXt bằng ignore_mismatched_sizes rồi mặc nhiên nghiệm thu. Exact resume sau đó dùng model/config ConvNeXt đã tạo.

In [ ]:
with RUN.gate('weights') as evidence:
    from huggingface_hub import snapshot_download
    CACHE=REPO/'runs/pretrained/hf-cache'
    detector_receipt=download_pretrained(CACHE,revision=CONFIG['detector_revision'],offline=CONFIG['offline'])
    DETECTOR=Path(detector_receipt['snapshot_dir'])
    CONVNEXT=Path(snapshot_download(repo_id=CONFIG['backbone_repo'],revision=CONFIG['backbone_revision'],
        allow_patterns=['config.json','model.safetensors'],cache_dir=str(CACHE),
        local_files_only=CONFIG['offline'],token=False)).resolve()
    assert CONVNEXT.name==CONFIG['backbone_revision']
    files={}
    for path in [CONVNEXT/'config.json',CONVNEXT/'model.safetensors',
                 DETECTOR/'config.json',DETECTOR/'preprocessor_config.json',DETECTOR/'model.safetensors']:
        assert path.is_file() and path.stat().st_size>0
        SOURCE_BEFORE[str(path)]=('sha256',sha256_file(path))
        files[str(path)]={'sha256':sha256_file(path),'bytes':path.stat().st_size}
    CONFIG.update(detector_dir=str(DETECTOR),backbone_file=str(CONVNEXT/'model.safetensors'))
    evidence.update(detector_receipt=detector_receipt,backbone_repo=CONFIG['backbone_repo'],
                    backbone_revision=CONFIG['backbone_revision'],files=files)
    save_json(ARTIFACT/'pretrained_files.json',evidence,indent=2)
    save_json(ARTIFACT/'config.json',CONFIG,indent=2)

## 5. CUDA smoke mỗi resolution — features, projections, load report, losses/resume

Dùng cùng module `exercise_baseline` của notebook 02 để kiểm tra losses, gradient, optimizer, frozen hashes, hai-pass predictions và controlled Lightning restore. Không đổi defaults của CLI/runner. Scoped factory chỉ đặt backbone/config của lần nghiệm thu; source detector được nạp local và core load được kiểm tra trước khi warm start.

Feature gate dùng batch thật và thêm vùng mask giả ở biên để kiểm tra maskdownsampling; train batch không bị thay đổi. Checkpoint gate tạo **model mới từ config đã lưu**, tắt pretrained reload để không tải weights lần nữa, load strict full state rồi so predictions/hash. OOM giữ kết quả resolution đã đạt, nhưng không tạo overall PASS.

In [ ]:
def make_args(output):
    args=pdp_main.get_args_parser().parse_args([
        '--task_config',str(task_config),'--n_classes','225','--repo_name','',
        '--n_tasks','5','--batch_size','1','--eff_batch_size','1','--num_workers','0',
        '--n_gpus','1','--accelerator','gpu','--epochs','2','--tf32','0','--augment','0',
        '--use_prompts','1','--local_query','1','--num_prompts','100','--prompt_len','10',
        '--lambda_query','0.1','--ddl_lambda','0.15','--query_loss_grad','1',
        '--freeze','backbone,encoder,decoder','--new_params','class_embed,prompts',
        '--optim_groups','prompt','--freeze_shared_after_task1','0','--pseudo_dedup_iou','0',
        '--lr','0.0001','--lr_old','0.00001','--output_dir',str(output)])
    pdp_main.setup_task_info(args)
    return args

for resolution in CONFIG['resolutions']:
    name=f'cuda_{resolution}'
    output=ARTIFACT/str(resolution);output.mkdir()
    module=None;source=None;reloaded=None;state=None;transfer=None;plan=None
    try:
        with RUN.gate(name) as evidence, (output/'model.log').open('w') as log:
            pl.seed_everything(CONFIG['seed'],workers=True)
            torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats()
            args=make_args(output);args.log_file=log
            source_profile=DeformableDetrConfig.from_pretrained(str(DETECTOR),local_files_only=True)
            assert source_profile.backbone=='resnet50' and source_profile.encoder_layers==source_profile.decoder_layers==6
            assert source_profile.d_model==256 and source_profile.num_queries==300 and source_profile.num_feature_levels==4
            source_profile.use_pretrained_backbone=False;source_profile.use_prompts=0;source_profile.local_query=0
            source,load_info=detr.DeformableDetrForObjectDetection.from_pretrained(str(DETECTOR),
                config=source_profile,local_files_only=True,output_loading_info=True)
            counters=[f'{n}.num_batches_tracked' for n,m in source.named_modules() if isinstance(m,detr.DeformableDetrFrozenBatchNorm2d)]
            validate_loading_info(load_info,frozen_batch_norm_counters=counters)
            save_json(output/'source_detector_loading.json',load_info,indent=2)
            state=source.state_dict()
            profile=DeformableDetrConfig.from_pretrained(str(DETECTOR),local_files_only=True)
            profile.backbone=CONFIG['backbone'];profile.use_pretrained_backbone=True
            profile.backbone_pretrained_file=CONFIG['backbone_file'];profile.backbone_feature_info=None
            profile.disable_custom_kernels=not native
            processor=DeformableDetrImageProcessor.from_pretrained(str(DETECTOR),local_files_only=True,
                size={'shortest_edge':resolution,'longest_edge':resolution})
            dataset=CocoDetection(str(RAW),str(TASKS/'train_task_1.json'),processor)
            subset=Subset(dataset,[dataset.ids.index(iid) for iid in CONFIG['image_ids']]);subset.collate_fn=dataset.collate_fn
            loader=pdp_main.make_train_loader(subset,args)
            probe=next(iter(loader))
            assert all(set(t['class_labels'].tolist())<=set(range(100)) for t in probe['labels'])
            loaded_backbone={}
            def audited_create(name,**kwargs):
                backbone=timm.create_model(name,**kwargs)
                if name==CONFIG['backbone'] and kwargs.get('pretrained'):
                    loaded_backbone['sha256']=state_digest(backbone.state_dict())
                return backbone
            with patch.object(engine,'DeformableDetrConfig',lambda:profile), patch.object(detr,'create_model',side_effect=audited_create):
                module=engine.local_trainer(loader,None,None,args,SimpleNamespace(),task_id=1)
            assert loaded_backbone and state_digest(module.model.model.backbone.conv_encoder.model.state_dict())==loaded_backbone['sha256'], 'Detector post_init overwrote pretrained backbone'
            save_json(output/'backbone_pretrained_preservation.json',loaded_backbone,indent=2)
            backbone_before=state_digest(module.model.model.backbone.state_dict())
            plan=plan_detector_transfer({n:tuple(t.shape) for n,t in state.items()},
                                       {n:tuple(t.shape) for n,t in module.model.state_dict().items()})
            transfer={n:state[n] for n in plan['load_keys']}
            info=module.model.load_state_dict(transfer,strict=False)
            assert set(info.missing_keys)==set(plan['new_target_keys']) and not info.unexpected_keys
            assert state_digest(module.model.model.backbone.state_dict())==backbone_before
            save_json(output/'detector_transfer_plan.json',plan,indent=2)
            del source,state,transfer;source=state=transfer=None;gc.collect()
            module.processor=processor;module.model.model.prompts.set_task_id(0)
            module.resume();module.model.model.prompts.init_task_prompts();module.to('cuda');module.eval()
            conv=module.model.model.backbone.conv_encoder
            assert conv.intermediate_channel_sizes==[256,512,1024]
            assert [r['stride'] for r in module.model.config.backbone_feature_info]==[8,16,32]
            assert len(module.model.model.input_proj)==4
            cfg=conv.model.pretrained_cfg
            np.testing.assert_allclose(processor.image_mean,cfg['mean'])
            np.testing.assert_allclose(processor.image_std,cfg['std'])
            masked=probe['pixel_mask'].clone();masked[:,-max(1,resolution//5):,:]=0;masked[:,:,-max(1,resolution//5):]=0
            with torch.no_grad():features=conv(probe['pixel_values'].to('cuda'),masked.to('cuda'))
            rows=[]
            for (feature,mask),description in zip(features,module.model.config.backbone_feature_info,strict=True):
                assert feature.shape[1]==description['channels'] and mask.shape==feature.shape[:1]+feature.shape[2:]
                assert feature.dtype==torch.float32 and mask.dtype==torch.bool
                assert torch.isfinite(feature).all() and mask[0,0,0] and not mask[0,-1,-1]
                expected=((resolution//description['stride']),(resolution//description['stride']))
                assert tuple(feature.shape[-2:])==expected
                rows.append({'feature':list(feature.shape),'mask':list(mask.shape),'stride':description['stride']})
            for index,(feature,mask) in enumerate(features):
                with torch.no_grad():projected=module.model.model.input_proj[index](feature)
                assert projected.shape[1]==256 and projected.shape[-2:]==feature.shape[-2:]
            save_json(output/'features.json',rows,indent=2)
            del features,feature,projected
            evidence.update(features=rows,backbone_initial_sha256=backbone_before,
                            detector_core_transferred=len(plan['load_keys']),new_keys=len(plan['new_target_keys']))
            projection_shapes={}
            def capture_projection(index):
                def capture(layer,inputs,value):
                    assert value.shape[1]==256 and torch.isfinite(value).all()
                    projection_shapes[index]=list(value.shape)
                return capture
            hooks=[layer.register_forward_hook(capture_projection(i)) for i,layer in enumerate(module.model.model.input_proj)]
            try:
                report=exercise_baseline(module,loader,args,output)
            finally:
                for hook in hooks:hook.remove()
            assert set(projection_shapes)=={0,1,2,3}
            for index,stride in enumerate([8,16,32,64]):
                assert projection_shapes[index][-2:]==[(resolution+stride-1)//stride]*2
            evidence.update(report,projection_shapes=projection_shapes)
            # Cold construction with no pretrained downloads, strict model-state load.
            module.to('cuda');module.eval()
            def predictions(model):
                with torch.no_grad():
                    result=predict_batch(model,probe['pixel_values'].to('cuda'),probe['pixel_mask'].to('cuda'),
                        torch.stack([t['orig_size'] for t in probe['labels']]).to('cuda'),
                        use_prompts=True,local_query=True,seen_classes=100)
                return [{k:v.cpu() for k,v in row.items()} for row in result]
            reference=predictions(module.model)
            final=torch.load(output/'task_final.pth',map_location='cpu')
            module.cpu();del module;module=None;gc.collect();torch.cuda.empty_cache()
            reload_profile=DeformableDetrConfig.from_pretrained(str(output/'resolved_model_config'),local_files_only=True)
            assert reload_profile.backbone==CONFIG['backbone'] and reload_profile.num_labels==225
            reload_profile.use_pretrained_backbone=False;reload_profile.backbone_pretrained_file=None
            with patch.object(engine,'DeformableDetrConfig',lambda:reload_profile):
                reloaded=engine.local_trainer(loader,None,None,args,SimpleNamespace(),task_id=1)
            reloaded.model.load_state_dict(final['model'],strict=True)
            reloaded.model.model.prompts.set_task_id(0);reloaded.to('cuda');reloaded.eval()
            assert state_digest(reloaded.model.state_dict())==state_digest(final['model'])
            actual=predictions(reloaded.model)
            for got,want in zip(actual,reference,strict=True):
                for key in want:
                    if want[key].is_floating_point():torch.testing.assert_close(got[key],want[key],rtol=1e-5,atol=1e-4)
                    else:assert torch.equal(got[key],want[key])
            evidence.update(cold_model_config_reload_equal=True,native_kernel_accepted=native_tested,
                peak_allocated_bytes=torch.cuda.max_memory_allocated(),peak_reserved_bytes=torch.cuda.max_memory_reserved(),
                resolved_config_sha256=sha256_file(output/'resolved_model_config/config.json'))
            save_json(output/'args.json',{k:v for k,v in vars(args).items() if k!='log_file'},indent=2)
        save_json(output/'result.json',RUN.data['gates'][name],indent=2)
    except torch.cuda.OutOfMemoryError as error:
        RUN.record(name,{'status':'OOM','error':str(error),'finished_at':now()})
        print(name,'OOM; overall acceptance remains partial')
    finally:
        module=source=reloaded=state=transfer=plan=None
        for variable in ['loader','dataset','subset','probe','final','conv','masked','mask','hooks']:
            if variable in globals():globals()[variable]=None
        gc.collect();torch.cuda.empty_cache()

## 6. Nghiệm thu cuối và train-readiness

Hash sources/configs/weights/raw/holdout sau smoke. PASS chỉ khi đủ 8 gates. `full_training_ready=False` luôn giữ vì notebook này chưa nối CLI/runner, chưa đo budget teacher Task 2+, chưa calibration và chưa kiểm tra instance Vast.ai. Native kernel được ghi riêng; fallback không được báo native PASS.

Sau khi chạy, lưu notebook có outputs theo run ID. Các artifact lớn không vào git. Nếu lỗi, đọc model.log/result và gate đang FAIL; không sửa một run cũ thành PASS.

In [ ]:
with RUN.gate('sources_preserved') as evidence:
    for filename,(algorithm,digest) in SOURCE_BEFORE.items():
        actual=md5_file(filename) if algorithm=='md5' else sha256_file(filename)
        assert actual==digest,f'Source changed: {filename}'
    evidence.update(checked_files=len(SOURCE_BEFORE),sources_holdout_and_weights_unchanged=True)
summary=RUN.finish()
RUN.data['convnext_accepted']=summary['status']=='PASS_CONVNEXT_ADAPTER_CUDA_SMOKE'
RUN.data['native_kernel_accepted']=RUN.data['convnext_accepted'] and all(
    RUN.data['gates'].get(f'cuda_{r}',{}).get('native_kernel_accepted',False) for r in [640,800])
RUN.data.update(full_training_ready=False,limitations=[
    'Task 1 smoke, not convergence or full five-task teacher/PPG acceptance.',
    'Training CLI/runner still require explicit backbone/weights/resolution integration.',
    'Vast.ai environment and native kernel acceptance are separate.',
    'Calibration policy and EXP-B1/B2 step/VRAM budget remain pending.'])
RUN.persist();save_json(ARTIFACT/'summary.json',RUN.data,indent=2)
save_json(ARTIFACT.parent/'latest.json',{'run_id':ARTIFACT.name,'status':RUN.data['status'],
    'summary':str(ARTIFACT/'summary.json'),'finished_at':RUN.data['finished_at']},indent=2)
print(json.dumps(RUN.data,ensure_ascii=False,indent=2))
print('Save notebook outputs; full training on Vast.ai is not accepted by this smoke alone.')